# Data preparation

This script was developed to bring in relevant environmental data for modeling carbon stock as a function of environmental co-variates from remote sensing and Copernicus data products.

Note: 
* The first function (extract_values) extracts values of matching sites, however this gives NAs for sites that appear over land given the lower resolution data products.
* The second function extract_closest_values() provides the closest match for any sites that received an NA.

## Input data:

* Seagrass site data, `Seagrass_site`, Seagrass_site_data.xlsx
* 95th percentile of bottom temperature (°C): `bottomT_p95_C_closest`, bottomT_p95_daily_C.nc
* Eastward seawater velocity (m s-1), `uo_mean_1.5m_m_s_closest`, uo_mean_1.5m_m_s.nc
* Northward seawater velocity (m s-1), `vo_p90_1.5m_m_s_closest`, vo_p90_1.5m_m_s.nc
* Phosphate at sub surface depth 1.5 m (mmol m-3), `po4_mean_1.5m_mmol_m3`, po4_mean_monthly_1.5m_mmol_m3.nc
* PH at sub surface depth 1.5 m (1), `pH_mean_1.5m`, pH_mean_monthly_1.5m.nc
* Sea surface wave significant height (m), `wave_height_VHM0_p95_m`, VHM0_p95_m.nc
* Surface downward flux of total CO2 (molC m-2 yr-1), `Surf_fgco2_p95_molC_m2_yr`, Surf_fgco2_p95_molC_m2_yr.nc
* Diffuse attenuation coefficient at 490 nm, `KD490`, S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km.nc
* Remote sensing reflectance at 443 nm, `RRS443`, S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_443.4km.nc


## Bottom_T_p95

Data comes from the Global Ocean Physics Reanalysis.
I used the daily product for bottom_T and the monthly product for the current data, [GLOBAL_MULTIYEAR_PHY_001_030](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/services)

* Input: `cmems_mod_glo_phy_my_0.083deg_P1D-m`, GLOBAL_MULTIYEAR_PHY_001_030, "bottomT" is Sea water potential temperature at sea floorbottomT [°C]
* Output: `bottomT_p95_daily_C.nc` is 95th percentile of bottom temperature at each location from daily physics product

### Load NetCDF

```R
bottomT_daily <- open.nc("data/cmems_mod_glo_phy_my_0.083deg_P1D-m_bottomT_10.00W-34.00E_34.00N-61.00N_1993-01-01-2021-06-30.nc")
print.nc(bottomT_daily)
```

### Load NetCDF as SpatRaster

```R
bottomT_daily_raster <- rast("data/cmems_mod_glo_phy_my_0.083deg_P1D-m_bottomT_10.00W-34.00E_34.00N-61.00N_1993-01-01-2021-06-30.nc", subds = "bottomT")

# Calculate the 95th percentile of bottom temperature at each location from the daily physics product
p95_bottomT_daily_C <- app(bottomT_daily_raster, fun = function(x) unname(quantile(x, probs = 0.95, na.rm = TRUE)))
plot(p95_bottomT_daily_C)

# Write this to a NetCDF for future use
writeCDF(p95_bottomT_daily_C, "bottomT_p95_daily_C.nc", varname = "p95_bottomT_daily_C", overwrite = TRUE)
```